# Chapter 2 — NB 03: anotação funcional da tabela união com **Biofilter 4** (pLOF + AlphaMissense)

**Objetivo:** anotar as variantes do `cmp_union_all_variants.csv` com **pLOF (LOFTEE)** e **AlphaMissense** via Biofilter 4 (BF4),
e — igualmente importante — **quantificar a cobertura e as lacunas do BF4** (o que ele NÃO responde), para mapear seus pontos fracos.
A comparação do que o BF4 perdeu (via VEP) fica para um momento futuro.

**Como o BF4 foi rodado (LSF):** `scripts/03/run_biofilter_annotation.sh` + `submit_biofilter_annotation.sh`
→ relatório `annotation_master_variant` (`most_severe_only=true`) sobre `chr:pos:ref:alt` das variantes da união.
Saída: `results/03/znf175_bf4_annot.csv`.

> ⚠️ Nota desde já: o DB do BF4 (build 20260514) é baseado em **variantes conhecidas (gnomAD)**.
> Variantes **privadas/ultra-raras** do PMBB tendem a sair como `not_found`.

In [1]:
# --- Setup ---
from pathlib import Path
import pandas as pd
import numpy as np

BASE      = Path("/project/hall/analysis/hearing-loss-genomics")
RESULTS   = BASE / "analysis/chapter_2/results/03"
RESULTS02 = BASE / "analysis/chapter_2/results/02"

union = pd.read_csv(RESULTS02 / "cmp_union_all_variants.csv")
bf4   = pd.read_csv(RESULTS / "znf175_bf4_annot.csv")
print("union variants:", len(union))
print("BF4 annotation rows (incl. multi-transcript):", len(bf4))

union variants: 386
BF4 annotation rows (incl. multi-transcript): 625


## 1. Deduplicar a anotação do BF4 (1 linha por variante)

Mesmo com `most_severe_only=true`, o BF4 pode retornar linhas repetidas por variante (transcritos empatados).
A saída já vem ordenada por severidade, então mantemos a **primeira** linha por `input_value`.

In [2]:
bf4_1 = bf4.drop_duplicates("input_value").copy()
print("unique variants annotated:", len(bf4_1))
print("status:"); print(bf4_1["status"].value_counts().to_string())

unique variants annotated: 384
status:
status
not_found    287
found         97


## 2. Anexar a anotação do BF4 à tabela união

Chave: `input_value` do BF4 = `19:` + chave da união (`POS:REF:ALT`).
Mantemos as colunas-chave: consequência, **pLOF (lof_confidence)**, **AlphaMissense**, REVEL/CADD/SpliceAI, gnomAD AF.

In [3]:
bf4_cols = ["input_value","status","gene_symbol","consequence_name","consequence_category",
            "impact_name","lof_confidence","lof_filter",
            "alphamissense_score","alphamissense_classification",
            "revel_max","cadd_phred","spliceai_ds_max","sift_max","polyphen_max",
            "af","grpmax","grpmax_af","hgvsp","amino_acids"]
bf4_cols = [c for c in bf4_cols if c in bf4_1.columns]

union = union.copy()
union["bf4_key"] = "19:" + union["key"]
ann = union.merge(bf4_1[bf4_cols], left_on="bf4_key", right_on="input_value", how="left")
ann["bf4_status"] = ann["status"].fillna("not_submitted")   # '*' alleles were not submitted to BF4

ann.to_csv(RESULTS / "cmp_union_annotated_bf4.csv", index=False)
print("saved -> results/03/cmp_union_annotated_bf4.csv  (", ann.shape[1], "columns )")
print("\nbf4_status:"); print(ann["bf4_status"].value_counts().to_string())

saved -> results/03/cmp_union_annotated_bf4.csv  ( 40 columns )

bf4_status:
bf4_status
not_found        287
found             97
not_submitted      2


## 3. Pontos fortes do BF4 — o que ele entregou

As variantes **found**: consequência VEP, pLOF (LOFTEE), e scores pré-computados.

In [4]:
found = ann[ann["bf4_status"]=="found"]
print(f"found: {len(found)} de {len(ann)} variantes\n")

print("consequence_name (found):")
print(found["consequence_name"].value_counts(dropna=False).to_string())

print("\npLOF (lof_confidence):")
print(found["lof_confidence"].value_counts(dropna=False).to_string())

print("\nVariantes pLOF identificadas pelo BF4:")
lof = found[found["lof_confidence"].notna()]
print(lof[["key","consequence_name","lof_confidence","MAF_v1","MAF_v2","af"]].to_string(index=False))

found: 97 de 386 variantes

consequence_name (found):
consequence_name
missense_variant         46
synonymous_variant       21
intron_variant           18
frameshift_variant        4
3_prime_UTR_variant       3
5_prime_UTR_variant       2
stop_gained               2
splice_region_variant     1

pLOF (lof_confidence):
lof_confidence
NaN    91
LC      5
HC      1

Variantes pLOF identificadas pelo BF4:
             key   consequence_name lof_confidence   MAF_v1   MAF_v2       af
  51573407:CAG:C frameshift_variant             LC 0.000044 0.000034 0.000085
   51581437:A:AG frameshift_variant             HC 0.000611 0.001098 0.001321
    51587583:C:T        stop_gained             LC      NaN 0.000057 0.000158
51587727:CAAAG:C frameshift_variant             LC 0.000131 0.000080 0.000085
    51587814:C:T        stop_gained             LC 0.000218 0.000206 0.000105
   51588428:CA:C frameshift_variant             LC 0.000306 0.000172 0.000158


## 4. ⚠️ Pontos fracos do BF4 — o que ele NÃO respondeu

Dois furos para o nosso objetivo:
1. **Cobertura** — variantes privadas/ultra-raras saem como `not_found`.
2. **AlphaMissense** — checamos se vem populado para o ZNF175.

In [5]:
# 4a. AlphaMissense coverage among found missense
miss = found[found["consequence_name"].astype(str).str.contains("missense")]
am_n = found["alphamissense_score"].notna().sum()
print(f"missense entre found: {len(miss)}")
print(f"variantes com alphamissense_score: {am_n}")
if am_n == 0:
    print(">>> AlphaMissense NAO esta populado para o ZNF175 nesta build do BF4 (0 scores).")

# 4b. found-rate por faixa de raridade (nossa MAF)
ann["maf_max"] = ann[["MAF_v1","MAF_v2"]].max(axis=1)
sub = ann[ann["ALT"]!="*"].copy()   # '*' nao foram submetidas
sub["found"] = (sub["bf4_status"]=="found").astype(int)
sub["bin"] = pd.cut(sub["maf_max"], [0,1e-4,1e-3,1e-2,1],
                    labels=["<0.01%","0.01-0.1%","0.1-1%",">=1%"])
cov = sub.groupby("bin", observed=True)["found"].agg(["mean","size"]).rename(
        columns={"mean":"found_rate","size":"n"})
cov["found_rate"] = cov["found_rate"].round(2)
print("\nfound-rate por faixa de raridade:")
print(cov.to_string())
cov.to_csv(RESULTS / "bf4_coverage_by_rarity.csv")

missense entre found: 46
variantes com alphamissense_score: 0
>>> AlphaMissense NAO esta populado para o ZNF175 nesta build do BF4 (0 scores).

found-rate por faixa de raridade:
           found_rate    n
bin                       
<0.01%           0.11  316
0.01-0.1%        0.90   41
0.1-1%           1.00   16
>=1%             1.00    8


In [6]:
# --- Summary ---
n_found   = int((ann["bf4_status"]=="found").sum())
n_notf    = int((ann["bf4_status"]=="not_found").sum())
n_star    = int((ann["ALT"]=="*").sum())
n_lof     = int(found["lof_confidence"].notna().sum())
n_lof_hc  = int((found["lof_confidence"]=="HC").sum())

summary = f'''# NB 03 — Biofilter 4 annotation of the ZNF175 union table

Report: annotation_master_variant (most_severe_only). Input: {len(ann)-n_star} variants (chr:pos:ref:alt; '*' excluded).
Output: cmp_union_annotated_bf4.csv (union table + BF4 columns).

## What BF4 delivered (strengths)
- found: {n_found} / {len(ann)} variants  (rich precomputed scores: REVEL, CADD, SpliceAI, gnomAD AF, VEP consequence)
- pLOF (LOFTEE): {n_lof} variants flagged ({n_lof_hc} HC + {n_lof-n_lof_hc} LC) -- frameshift / stop_gained

## What BF4 did NOT answer (weaknesses — to revisit with VEP)
- not_found: {n_notf} variants ({round(100*n_notf/max(1,len(ann)-n_star))}% of submitted). ROOT CAUSE: the loaded
  BF4 DB was filtered to **gnomAD AC >= 5** (a build-time choice) -> variants with AC < 5 are excluded. They DO
  exist in gnomAD (10 not_found spot-checked on the gnomAD site -> all AC < 5); BF4-found variants have min ac = 5.
  So this is a CONFIGURABLE cutoff, not an inherent gnomAD limit -> reversible by reloading BF4 without the AC filter.
  (low cohort-MAF tracks low gnomAD AC, hence found-rate ~11% in the <0.01% bin.)
- AlphaMissense: NOT populated for ZNF175 in this DB build -> 0 scores despite found missense variants.
- '*' spanning-deletion alleles ({n_star}) not submitted (not real alleles).

## Implication
BF4 is strong for variants ABOVE its AC cutoff (pLOF + precomputed scores). To include the rare/private burden
candidates, either reload BF4 without the AC filter, or annotate by coordinate. NEXT: run VEP + dbNSFP4.5a on the
region VCF (annotates ALL variants) and compare against BF4 to quantify exactly what BF4's AC cutoff excluded.

## Outputs
- cmp_union_annotated_bf4.csv, bf4_coverage_by_rarity.csv, znf175_bf4_annot.csv
'''
(RESULTS / "nb03_bf4_annotation_summary.md").write_text(summary)
print(summary)

# NB 03 — Biofilter 4 annotation of the ZNF175 union table

Report: annotation_master_variant (most_severe_only). Input: 384 variants (chr:pos:ref:alt; '*' excluded).
Output: cmp_union_annotated_bf4.csv (union table + BF4 columns).

## What BF4 delivered (strengths)
- found: 97 / 386 variants  (rich precomputed scores: REVEL, CADD, SpliceAI, gnomAD AF, VEP consequence)
- pLOF (LOFTEE): 6 variants flagged (1 HC + 5 LC) -- frameshift / stop_gained

## What BF4 did NOT answer (weaknesses — to revisit with VEP)
- not_found: 287 variants (75% of submitted). ROOT CAUSE: the loaded
  BF4 DB was filtered to **gnomAD AC >= 5** (a build-time choice) -> variants with AC < 5 are excluded. They DO
  exist in gnomAD (10 not_found spot-checked on the gnomAD site -> all AC < 5); BF4-found variants have min ac = 5.
  So this is a CONFIGURABLE cutoff, not an inherent gnomAD limit -> reversible by reloading BF4 without the AC filter.
  (low cohort-MAF tracks low gnomAD AC, hence found-rate ~11% in the <0